# Normalização dos dados e camada processed

Transforma os dados brutos de `data/raw/` em uma camada tratada em `data/processed/`, que será a base comum para todas as análises do grupo.

Etapas:
1. Converter as colunas de data para o tipo data.
2. Padronizar os textos: maiúsculas, sem acentos, sem espaços nas pontas e sem espaços duplos.
3. Remover as linhas duplicadas da geolocalização.
4. Criar indicadores no nível do pedido: valores, atraso, avaliação, localização do cliente e mês da compra.
5. Salvar as tabelas tratadas.

A única remoção de linhas é a de duplicatas exatas da geolocalização, que não trazem informação nova. Todos os pedidos, inclusive cancelados e não entregues, são mantidos: o filtro por status é aplicado em cada análise conforme a necessidade (ex.: receita considera apenas pedidos `DELIVERED`).

## Bibliotecas

In [1]:
%run ../../src/librarys.ipynb

## Carga dos dados brutos

Mesma leitura do inventário: CEPs como texto (para manter o zero à esquerda) e `utf-8-sig` na tabela de tradução (por causa do caractere invisível no início do arquivo).

In [2]:
customers = pd.read_csv("../../data/raw/olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
orders = pd.read_csv("../../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../../data/raw/olist_products_dataset.csv")
category_translation = pd.read_csv("../../data/raw/product_category_name_translation.csv", encoding="utf-8-sig")
sellers = pd.read_csv("../../data/raw/olist_sellers_dataset.csv", dtype={"seller_zip_code_prefix": str})
geolocation = pd.read_csv("../../data/raw/olist_geolocation_dataset.csv", dtype={"geolocation_zip_code_prefix": str})

## 1. Conversão das datas

As datas foram lidas como texto. Convertê-las para o tipo data permite calcular diferenças entre elas (ex.: dias de atraso) e agrupar por mês. Isso é feito antes da padronização de textos para que as datas não sejam tratadas como texto.

In [3]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 2. Padronização dos textos

Textos escritos de formas diferentes (`sao paulo`, `Sao Paulo`, `sao  paulo`) seriam tratados como valores distintos em agrupamentos e joins. A função abaixo aplica a mesma regra a todas as colunas de texto de uma tabela:

- converte para **maiúsculas**;
- remove **espaços no início e no fim**;
- troca **dois ou mais espaços seguidos** por um só.

As colunas de identificador (terminadas em `_id`) ficam de fora: são códigos, não textos, e precisam continuar idênticos aos dados originais.

In [4]:
def normalizar_texto(df):
    for coluna in df.select_dtypes(include="str").columns:
        if coluna.endswith("_id"):
            continue
        df[coluna] = (
            df[coluna]
            .str.upper()
            .str.normalize("NFKD")
            .str.encode("ascii", "ignore")
            .str.decode("ascii")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    return df

Exemplo antes da padronização: cidades da geolocalização com espaços duplos ou espaço no fim do nome.

In [5]:
geolocation[geolocation["geolocation_city"].str.contains("  ") | geolocation["geolocation_city"].str.endswith(" ")]["geolocation_city"].value_counts().head(10)

geolocation_city
itapejara d  oeste         6
diamante d  oeste          2
salvador                   1
barra de  santo antônio    1
pau d  arco                1
rancho alegre d  oeste     1
Name: count, dtype: int64

Exemplo antes da padronização: a mesma cidade escrita com e sem acento.

In [6]:
geolocation[geolocation["geolocation_city"].isin(["sao paulo", "são paulo"])]["geolocation_city"].value_counts()

geolocation_city
sao paulo    135800
são paulo     24918
Name: count, dtype: int64

Aplica a padronização em todas as tabelas.

In [7]:
customers = normalizar_texto(customers)
orders = normalizar_texto(orders)
order_items = normalizar_texto(order_items)
payments = normalizar_texto(payments)
reviews = normalizar_texto(reviews)
products = normalizar_texto(products)
category_translation = normalizar_texto(category_translation)
sellers = normalizar_texto(sellers)
geolocation = normalizar_texto(geolocation)

Resultado da padronização em algumas tabelas.

In [8]:
geolocation[geolocation["geolocation_city"] == "SAO PAULO"]["geolocation_city"].value_counts()

geolocation_city
SAO PAULO    160719
Name: count, dtype: int64

In [9]:
customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,FRANCA,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,SAO BERNARDO DO CAMPO,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,SAO PAULO,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,MOGI DAS CRUZES,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,CAMPINAS,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,JARAGUA DO SUL,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,SAO PAULO,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,TIMOTEO,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,CURITIBA,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,BELO HORIZONTE,MG


In [10]:
orders["order_status"].value_counts()

order_status
DELIVERED      96478
SHIPPED         1107
CANCELED         625
UNAVAILABLE      609
INVOICED         314
PROCESSING       301
CREATED            5
APPROVED           2
Name: count, dtype: int64

In [11]:
products.head(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,PERFUMARIA,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,ARTES,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,ESPORTE_LAZER,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,BEBES,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,UTILIDADES_DOMESTICAS,37.0,402.0,4.0,625.0,20.0,17.0,13.0
5,41d3672d4792049fa1779bb35283ed13,INSTRUMENTOS_MUSICAIS,60.0,745.0,1.0,200.0,38.0,5.0,11.0
6,732bd381ad09e530fe0a5f457d81becb,COOL_STUFF,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0
7,2548af3e6e77a690cf3eb6368e9ab61e,MOVEIS_DECORACAO,56.0,184.0,2.0,900.0,40.0,8.0,40.0
8,37cc742be07708b53a98702e77a21a02,ELETRODOMESTICOS,57.0,163.0,1.0,400.0,27.0,13.0,17.0
9,8c92109888e8cdf9d66dc7e463025574,BRINQUEDOS,36.0,1156.0,1.0,600.0,17.0,10.0,12.0


## 3. Remoção de duplicatas da geolocalização

A geolocalização tem linhas inteiramente repetidas (mesmo CEP, coordenadas, cidade e UF). Após a padronização dos textos, linhas que diferiam só por acento ou maiúscula (`são paulo` / `SAO PAULO`) também passam a ser iguais. Essas linhas não trazem informação nova e só aumentam o tamanho do arquivo.

Os demais registros são mantidos: um mesmo CEP continua com várias coordenadas diferentes, que poderão ser consolidadas (ex.: pela média) quando alguém for usar mapas.

In [12]:
linhas_antes = len(geolocation)
geolocation = geolocation.drop_duplicates()

print("Linhas antes:", linhas_antes)
print("Linhas depois:", len(geolocation))
print("Removidas:", linhas_antes - len(geolocation))

Linhas antes: 1000163
Linhas depois: 720494
Removidas: 279669


## 4. Indicadores do pedido

Cria uma tabela de pedidos enriquecida (`orders_enriched`), com uma linha por pedido e as colunas abaixo:

| Coluna | Descrição |
|---|---|
| `order_price` | Soma do preço dos itens do pedido |
| `order_freight` | Soma do frete dos itens do pedido |
| `order_total` | Valor total do pedido: preço + frete |
| `review_score` | Média das avaliações do pedido |
| `customer_state` | UF do cliente |
| `customer_city` | Cidade do cliente |
| `purchase_year_month` | Ano e mês da compra (ex.: `2017-11`), para agregações mensais |
| `dt_diff` | Dias entre a data de entrega e a data estimada. Positivo = atrasou; negativo = chegou antes |
| `is_late` | 1 se o pedido foi entregue com atraso, 0 se no prazo |

### `order_price`, `order_freight` e `order_total`

Um pedido pode ter vários itens. Somamos preço e frete de todos os itens de cada pedido. Os dois valores são mantidos separados, porque a análise de frete precisa deles, e somados em `order_total`.

In [13]:
total_por_pedido = order_items.groupby("order_id")[["price", "freight_value"]].sum().reset_index()
total_por_pedido.columns = ["order_id", "order_price", "order_freight"]
total_por_pedido["order_total"] = total_por_pedido["order_price"] + total_por_pedido["order_freight"]

total_por_pedido.head(10)

,order_id,order_price,order_freight,order_total
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


### `review_score`

Alguns pedidos têm mais de uma avaliação (visto no inventário). Usamos a média para ficar com um único valor por pedido.

In [14]:
review_por_pedido = reviews.groupby("order_id")["review_score"].mean().reset_index()
review_por_pedido.head(10)

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
5,00048cc3ae777c65dbb7d2a0634bc1ea,4.0
6,00054e8431b9d7675808bcb819fb4a32,4.0
7,000576fe39319847cbb9d288c5617fa6,5.0
8,0005a1a1728c9d785b8e2b08b904576c,1.0
9,0005f50442cb953dcd1d21e1fb923495,4.0


### Junção com pedidos

Junta pedidos com o total, a avaliação e a cidade/UF do cliente. Usamos `how="left"` para manter **todos os pedidos**, mesmo os que não têm itens ou avaliação (nesses casos a coluna fica vazia).

In [15]:
orders_enriched = orders.merge(total_por_pedido, on="order_id", how="left")
orders_enriched = orders_enriched.merge(review_por_pedido, on="order_id", how="left")
orders_enriched = orders_enriched.merge(customers[["customer_id", "customer_state", "customer_city"]], on="customer_id", how="left")

print("Pedidos antes:", len(orders))
print("Pedidos depois:", len(orders_enriched))

Pedidos antes: 99441
Pedidos depois: 99441


### `purchase_year_month`

Ano e mês da compra no formato `AAAA-MM`. Facilita as agregações mensais (evolução de pedidos, receita e ticket médio) sem precisar converter a data em cada análise.

In [16]:
orders_enriched["purchase_year_month"] = orders_enriched["order_purchase_timestamp"].dt.strftime("%Y-%m")

orders_enriched["purchase_year_month"].value_counts().sort_index()

purchase_year_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64

### `dt_diff` e `is_late`

`dt_diff` é a diferença em dias entre a entrega real e a data estimada. Comparamos apenas as datas (sem o horário), porque a data estimada não tem hora: um pedido entregue às 15h do dia previsto está no prazo.

`is_late` vale 1 quando `dt_diff` é maior que zero. Pedidos que **não foram entregues** (em trânsito, cancelados) ficam vazios nas duas colunas: não é possível dizer se atrasaram.

In [17]:
data_entrega = orders_enriched["order_delivered_customer_date"].dt.normalize()
data_estimada = orders_enriched["order_estimated_delivery_date"].dt.normalize()

orders_enriched["dt_diff"] = (data_entrega - data_estimada).dt.days

orders_enriched["is_late"] = (orders_enriched["dt_diff"] > 0).astype("Int64")
orders_enriched.loc[orders_enriched["dt_diff"].isna(), "is_late"] = pd.NA

Conferência: quantidade de pedidos atrasados, no prazo e sem entrega.

In [18]:
orders_enriched["is_late"].value_counts(dropna=False)

is_late
0       89941
1        6535
<NA>     2965
Name: count, dtype: Int64

Distribuição de `dt_diff`. A mediana negativa indica que a maior parte dos pedidos chega antes da data estimada.

In [19]:
orders_enriched["dt_diff"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: dt_diff, dtype: float64

Resultado final da tabela de pedidos enriquecida.

In [20]:
orders_enriched[[
    "order_id", "order_status", "order_purchase_timestamp", "purchase_year_month",
    "order_delivered_customer_date", "order_estimated_delivery_date", "dt_diff", "is_late",
    "order_price", "order_freight", "order_total", "review_score", "customer_state", "customer_city",
]].head(10)

,order_id,order_status,order_purchase_timestamp,purchase_year_month,order_delivered_customer_date,order_estimated_delivery_date,dt_diff,is_late,order_price,order_freight,order_total,review_score,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,DELIVERED,2017-10-02 10:56:33,2017-10,2017-10-10 21:25:13,2017-10-18,-8.0,0,29.99,8.72,38.71,4.0,SP,SAO PAULO
1,53cdb2fc8bc7dce0b6741e2150273451,DELIVERED,2018-07-24 20:41:37,2018-07,2018-08-07 15:27:45,2018-08-13,-6.0,0,118.70,22.76,141.46,4.0,BA,BARREIRAS
2,47770eb9100c2d0c44946d9cf07ec65d,DELIVERED,2018-08-08 08:38:49,2018-08,2018-08-17 18:06:29,2018-09-04,-18.0,0,159.90,19.22,179.12,5.0,GO,VIANOPOLIS
3,949d5b44dbf5de918fe9c16f97b45f8a,DELIVERED,2017-11-18 19:28:06,2017-11,2017-12-02 00:28:42,2017-12-15,-13.0,0,45.00,27.20,72.20,5.0,RN,SAO GONCALO DO AMARANTE
4,ad21c59c0840e6cb83a9ceb5573f8159,DELIVERED,2018-02-13 21:18:39,2018-02,2018-02-16 18:17:02,2018-02-26,-10.0,0,19.90,8.72,28.62,5.0,SP,SANTO ANDRE
5,a4591c265e18cb1dcee52889e2d8acc3,DELIVERED,2017-07-09 21:57:05,2017-07,2017-07-26 10:57:55,2017-08-01,-6.0,0,147.90,27.36,175.26,4.0,PR,CONGONHINHAS
6,136cce7faa42fdb2cefd53fdc79a6098,INVOICED,2017-04-11 12:22:08,2017-04,NaT,2017-05-09,NaN,<NA>,49.90,16.05,65.95,2.0,RS,SANTA ROSA
7,6514b8ad8028c9f2cc2374ded245783f,DELIVERED,2017-05-16 13:10:30,2017-05,2017-05-26 12:55:51,2017-06-07,-12.0,0,59.99,15.17,75.16,5.0,RJ,NILOPOLIS
8,76c6e866289321a7c93b82b54852dc33,DELIVERED,2017-01-23 18:29:09,2017-01,2017-02-02 14:08:10,2017-03-06,-32.0,0,19.90,16.05,35.95,1.0,RS,FAXINALZINHO
9,e69bfb5eb88e0ed6a785585b27e16dbf,DELIVERED,2017-07-29 11:55:02,2017-07,2017-08-16 17:14:30,2017-08-23,-7.0,0,149.99,19.77,169.76,5.0,SP,SOROCABA


In [21]:
orders_enriched[["order_price", "order_freight", "order_total", "review_score", "dt_diff", "is_late"]].isna().sum()

order_price       775
order_freight     775
order_total       775
review_score      768
dt_diff          2965
is_late          2965
dtype: int64

## 5. Salvar a camada processed

Grava as tabelas padronizadas e a tabela de pedidos enriquecida em `data/processed/`. Os demais notebooks do projeto devem ler daqui, e não de `data/raw/`.

In [22]:
customers.to_csv("../../data/processed/customers.csv", index=False)
orders.to_csv("../../data/processed/orders.csv", index=False)
order_items.to_csv("../../data/processed/order_items.csv", index=False)
payments.to_csv("../../data/processed/payments.csv", index=False)
reviews.to_csv("../../data/processed/reviews.csv", index=False)
products.to_csv("../../data/processed/products.csv", index=False)
category_translation.to_csv("../../data/processed/category_translation.csv", index=False)
sellers.to_csv("../../data/processed/sellers.csv", index=False)
geolocation.to_csv("../../data/processed/geolocation.csv", index=False)

orders_enriched.to_csv("../../data/processed/orders_enriched.csv", index=False)